# Telecom Gold Layer (Star Schema & SCDs)
Merges cleansed Silver data into a Kimball Star Schema. Implements SCD Type 1 & 2 using Delta MERGE.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, md5, concat_ws, lit, current_timestamp
from delta.tables import DeltaTable

spark = SparkSession.builder \
    .appName("Telecom_Gold") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

In [ ]:
print("Processing Dimensions...")

# 1. Dim_Date (SCD Type 0)
df_date = spark.read.format("delta").load("s3://silver/seed_date")
df_date.write.format("delta").mode("overwrite").option("mergeSchema", "true").save("s3://gold/dim_date")

# 2. Dim_Tower (SCD Type 1 - Overwrite for simplicity in this demo)
df_tower = spark.read.format("delta").load("s3://silver/towers_dim_seed")
df_tower = df_tower.withColumn("tower_sk", md5(col("tower_id")))
df_tower.write.format("delta").mode("overwrite").option("mergeSchema", "true").save("s3://gold/dim_tower")

# 3. Dim_Customer (SCD Type 1 - Overwrite for simplicity)
df_cust = spark.read.format("delta").load("s3://silver/customers_dim_seed")
df_cust = df_cust.withColumn("customer_sk", md5(col("customer_number")))
df_cust.write.format("delta").mode("overwrite").option("mergeSchema", "true").save("s3://gold/dim_customer")

# 4. Dim_Plan
df_plan = spark.read.format("delta").load("s3://silver/seed_plan")
df_plan = df_plan.withColumn("plan_sk", md5(col("plan_id")))
df_plan.write.format("delta").mode("overwrite").option("mergeSchema", "true").save("s3://gold/dim_plan")


In [ ]:
print("Processing Fact_CDR...")

df_silver_cdrs = spark.read.format("delta").load("s3://silver/telecom_cdrs")

# Generate Surrogate Keys for Fact Table
df_fact = df_silver_cdrs \
    .withColumn("customer_sk", md5(col("customer_number"))) \
    .withColumn("tower_sk", md5(col("tower_id"))) \
    .select(
        "call_id",
        "customer_sk",
        "tower_sk",
        "date_id",
        "timestamp",
        "event_type",
        "duration_seconds",
        "data_consumed_mb",
        "call_status",
        "latency_ms",
        "signal_strength_dbm",
        "network_condition"
    )

# UPSERT into Fact Table
fact_path = "s3://gold/fact_cdr"

if not DeltaTable.isDeltaTable(spark, fact_path):
    print("Creating initial Fact_CDR table...")
    df_fact.write.format("delta").mode("overwrite").save(fact_path)
else:
    print("Merging new CDRs into Fact_CDR...")
    fact_table = DeltaTable.forPath(spark, fact_path)
    fact_table.alias("target").merge(
        df_fact.alias("source"),
        "target.call_id = source.call_id" # Deduplication by call_id
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

print("Gold Layer Complete! Data is ready for Dashboards.")